# Week 8: time-frequency analysis

**What we do.** We analyse a signal whose frequency changes in time (a chirp with a short click) with two tools: the short-time Fourier transform (STFT, Gabor transform) with a Gaussian window, and the continuous wavelet transform (CWT) with the Morlet wavelet.

**What to expect.** Both pictures show the rising frequency of the chirp as a bright diagonal curve. The STFT has the same resolution everywhere, fixed by the window length: a short window localizes the click well but smears frequencies, a long window the opposite. The CWT adapts: at high frequencies it uses short wavelets (good time resolution), at low frequencies long ones (good frequency resolution).

**How to read the code.** Functions come from `wavelets_lab.timefreq`. The signal is sampled at `rate = 1000` Hz, so time is in seconds and frequency in hertz. Spectrograms and scalograms show energy (`|coefficient|²`) as colour: bright = much energy at that time and frequency.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import timefreq

## The test signal

**What we do.** We create a chirp whose frequency increases linearly from 20 Hz to 300 Hz over about 4 seconds and add a short click (10 samples = 10 ms) at `t = 2` s.

**What to expect.** In the time domain the oscillations visibly get faster; the click is a narrow spike. The Fourier transform of the whole signal would show all frequencies from 20 to 300 Hz but not *when* they occur.

**How to read the code.** `timefreq.chirp(t, f0, f1)` returns `cos` of a quadratic phase. `signal[2000:2010] += 3.0` adds 3 to ten consecutive samples. The second panel zooms into the first 0.5 s.

In [ ]:
rate = 1000.0
t = np.arange(4096) / rate
signal = timefreq.chirp(t, 20.0, 300.0)
signal[2000:2010] += 3.0  # a short click at t = 2 s
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(t, signal)
axes[0].set_title("whole signal")
axes[1].plot(t, signal)
axes[1].set_xlim(0.0, 0.5)
axes[1].set_title("first 0.5 s")
for ax in axes:
    ax.set_xlabel("time [s]")
    ax.set_ylabel("amplitude")
plt.tight_layout()
plt.show()

## The Gaussian window

**What we do.** We draw Gaussian windows of length 64 and 512 samples.

**What to expect.** Bell curves of maximum 1; the longer window covers 0.512 s instead of 0.064 s. The STFT multiplies the signal by such a window shifted in time and takes the FFT of the product.

**How to read the code.** `timefreq.gaussian_window(length)` uses the standard deviation `length / 6`; dividing the sample index by `rate` converts it to seconds.

In [ ]:
plt.figure(figsize=(8, 3))
for length in (64, 512):
    window = timefreq.gaussian_window(length)
    plt.plot(
        np.arange(length) / rate,
        window,
        label=f"length {length} ({length / rate:.3f} s)",
    )
plt.xlabel("time within the window [s]")
plt.ylabel("window value")
plt.legend()
plt.show()

## STFT: one window size for all frequencies

**What we do.** We compute the spectrogram `|STFT|²` with a short (64 samples) and a long (512 samples) window.

**What to expect.** Short window: the click at 2 s is a sharp vertical line, but the chirp curve is thick (poor frequency resolution, about `rate / 64 ≈ 16` Hz). Long window: the chirp is a thin line, but the click is smeared over half a second. You cannot have both — the Heisenberg uncertainty principle.

**How to read the code.** `timefreq.stft(signal, window, hop, rate)` returns the frequencies, the times and the coefficients; `hop = 16` samples between consecutive windows. `timefreq.power` squares the modulus and `timefreq.plot_time_frequency` draws it with labelled axes (time in seconds, frequency in hertz).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, length in zip(axes, (64, 512), strict=True):
    freqs, times, coeffs = timefreq.stft(
        signal, timefreq.gaussian_window(length), 16, rate
    )
    timefreq.plot_time_frequency(
        ax, times, freqs, timefreq.power(coeffs), f"spectrogram, window {length}"
    )
    ax.set_ylim(0.0, 400.0)
plt.tight_layout()
plt.show()

## Choosing scales for the CWT

**What we do.** We choose 80 frequencies between 10 Hz and 400 Hz, equally spaced on a logarithmic scale, and convert them to scales of the Morlet wavelet.

**What to expect.** Scale and frequency are inversely proportional (`f = f_c · rate / a`, with the Morlet centre frequency `f_c ≈ 0.81`): small scales correspond to high frequencies, so the curve decreases.

**How to read the code.** `np.geomspace` gives geometrically spaced numbers; `timefreq.scale_for_frequency(frequencies, "morl", rate)` returns the scales. Both axes of the plot are logarithmic, so the relation `a ∝ 1/f` is a straight line of slope −1.

In [ ]:
frequencies = np.geomspace(10.0, 400.0, 80)
scales = timefreq.scale_for_frequency(frequencies, "morl", rate)
plt.figure(figsize=(6, 4))
plt.loglog(frequencies, scales)
plt.xlabel("frequency [Hz]")
plt.ylabel("Morlet scale a")
plt.show()

## CWT: short windows for high frequencies, long ones for low

**What we do.** We compute the scalogram `|CWT|²` of the signal for these scales.

**What to expect.** The chirp is a curve rising from 20 to 300 Hz. The click is a vertical line that becomes **sharper towards high frequencies** (short wavelets) and wider towards low frequencies (long wavelets). On the logarithmic frequency axis all frequencies get the same relative resolution.

**How to read the code.** `timefreq.cwt(signal, scales, "morl", rate)` returns the coefficients (one row per scale) and the frequency of each scale; `ax.set_yscale("log")` makes the frequency axis logarithmic.

In [ ]:
coeffs, freqs = timefreq.cwt(signal, scales, "morl", rate)
fig, ax = plt.subplots(figsize=(8, 5))
timefreq.plot_time_frequency(ax, t, freqs, timefreq.power(coeffs), "Morlet scalogram")
ax.set_yscale("log")
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.timefreq` (`src/wavelets_lab/timefreq.py`).

**How to read the code.** `rate` is the sampling rate in samples per second; all time axes are in seconds and frequency axes in hertz.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.arange(n) / rate` | sampling instants in seconds | integer → array |
| `np.geomspace(a, b, n)` | `n` numbers from `a` to `b` with constant ratio | numbers → array |
| `ax.set_ylim`, `ax.set_xlim` | visible range of an axis | numbers → – |
| `ax.set_yscale("log")` | logarithmic vertical axis | – |
| `plt.loglog` | graph with logarithmic axes | arrays → line |

### Module `wavelets_lab.timefreq` (file `src/wavelets_lab/timefreq.py`)

Time-frequency analysis: the short-time Fourier transform (STFT, Gabor transform) and the continuous wavelet transform (CWT, computed with PyWavelets). Imports:
```python
import numpy as np
import pywt
from matplotlib.axes import Axes
```

#### `timefreq.chirp(t, f0, f1)`

**What it is.** A linear chirp: a cosine whose instantaneous frequency grows linearly from `f0` at `t[0]` to `f1` at `t[-1]`. The phase is the integral of the frequency, `φ(s) = 2π (f0 s + (f1 - f0) s^2 / (2T))` with `s = t - t[0]` and `T = t[-1] - t[0]`.

**Input.** `t` (array-like, increasing): time instants; `f0`, `f1` (`float`): start and end frequencies.

**Output.** `float64` array `cos(φ)`.

**How to write it yourself.**
1. `shifted = t - t[0]`, `duration = shifted[-1]`.
2. `phase = f0 * shifted + (f1 - f0) * shifted**2 / (2 * duration)`.
3. Return `np.cos(2 * np.pi * phase)`.

#### `timefreq.gaussian_window(length, sigma=None)`

**What it is.** A Gaussian (Gabor) window `exp(-u^2 / (2σ^2))` centred in the middle of `length` samples, with maximum 1. The Gaussian has the best possible joint time-frequency localization (Heisenberg's inequality is an equality).

**Input.** `length` (`int`): number of samples; `sigma` (`float` or `None`): width in samples, `length / 6` by default (so the window is close to 0 at its ends).

**Output.** `float64` array of length `length`.

**How to write it yourself.**
1. `width = length / 6 if sigma is None else sigma`.
2. Offsets from the centre: `u = np.arange(length) - (length - 1) / 2`.
3. Return `np.exp(-u**2 / (2 * width**2))`.

#### `timefreq.stft(x, window, hop, rate=1.0)`

**What it is.** The short-time Fourier transform: the signal is cut into segments of the window's length starting every `hop` samples, each segment is multiplied by the window and transformed with the FFT. Column `j` of the result is the spectrum of the signal near time `times[j]`. The time and frequency resolutions are the same for all frequencies and are set by the window length: a long window resolves frequencies well but blurs time, a short one the opposite.

**Input.**
- `x` (array-like): real 1D signal.
- `window` (array-like): analysis window (not longer than the signal).
- `hop` (`int`, positive): shift between segments in samples.
- `rate` (`float`, default 1): sampling rate, used for the axes.

**Output.** Tuple `(frequencies, times, coefficients)`: frequencies of the rows (from 0 to `rate / 2`), centres of the segments, and the complex array of shape `(len(frequencies), len(times))`. Raises `ValueError` for a too long window or `hop <= 0`.

**How to write it yourself.**
1. Convert, check that `len(window) <= len(x)` and `hop > 0`.
2. Segment starts: `starts = np.arange(0, len(x) - len(window) + 1, hop)`.
3. Windowed segments as rows: `segments = np.stack([x[s : s + len(window)] * window for s in starts])`.
4. FFT of real data along each row, keeping non-negative frequencies: `np.fft.rfft(segments, axis=1)`; transpose (`.T`) so that rows are frequencies and columns are times.
5. Axes: `frequencies = np.fft.rfftfreq(len(window), d=1 / rate)`, `times = (starts + (len(window) - 1) / 2) / rate` (segment centres).

#### `timefreq.power(coefficients)`

**What it is.** The squared modulus `|c|^2` of STFT or CWT coefficients: the spectrogram or the scalogram, i.e. the distribution of energy in the time-frequency plane.

**Input.** `coefficients` (array-like, real or complex).

**Output.** `float64` array of the same shape.

**How to write it yourself.** Return `np.abs(coefficients) ** 2`.

#### `timefreq.plot_time_frequency(ax, times, frequencies, values, title=None, xlabel="time [s]", ylabel="frequency [Hz]")`

**What it is.** Draws a spectrogram or scalogram as a colour map (brighter colour = more energy) with labelled axes.

**Input.**
- `ax` (`Axes`): where to draw.
- `times` (array-like): time of every column; `frequencies` (array-like): frequency of every row.
- `values` (array-like): array of shape `(len(frequencies), len(times))`, e.g. from `power`.
- `title`, `xlabel`, `ylabel` (`str`): texts.

**Output.** The `Axes`.

**How to write it yourself.**
1. `ax.pcolormesh(times, frequencies, values, shading="auto")` draws a cell for every value at the given coordinates (it also works for non-uniform frequencies, e.g. a log scale).
2. `ax.set_xlabel(xlabel)`, `ax.set_ylabel(ylabel)`, and `ax.set_title(title)` if a title is given.
3. Return `ax`.

#### `timefreq.scale_for_frequency(frequencies, wavelet="morl", rate=1.0)`

**What it is.** The inverse of the scale-to-frequency relation `f = f_c * rate / a`, where `f_c` is the centre frequency of the wavelet. It lets you choose scales that correspond to the frequencies you want to see.

**Input.** `frequencies` (array-like, positive), `wavelet` (`str`), `rate` (`float`).

**Output.** `float64` array of scales.

**How to write it yourself.**
1. `centre = pywt.central_frequency(wavelet)` (about 0.8125 for Morlet).
2. Return `centre * rate / frequencies`.

#### `timefreq.cwt(x, scales, wavelet="morl", rate=1.0)`

**What it is.** The continuous wavelet transform `W(a, b) = (1/sqrt a) ∫ x(t) ψ((t - b)/a) dt` evaluated at the given scales `a` and at every sample `b`, computed with `pywt.cwt`. Small scales see high frequencies with short windows, large scales low frequencies with long windows.

**Input.**
- `x` (array-like): 1D signal.
- `scales` (array-like): positive scales.
- `wavelet` (`str`): continuous wavelet, e.g. `"morl"` (Morlet) or `"mexh"` (Mexican hat).
- `rate` (`float`): sampling rate.

**Output.** Tuple `(coefficients, frequencies)`: array of shape `(len(scales), len(x))` and the pseudo-frequency of each scale.

**How to write it yourself.**
1. Call `pywt.cwt(x, scales, wavelet, sampling_period=1 / rate)`; it returns the coefficients and the frequencies.
2. Return both as NumPy arrays.